## Import libs 

In [1]:
import os
import json
import torch
import re

from PIL import Image, ImageDraw
from pdf2image import convert_from_path
import pandas as pd

from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from surya.layout import LayoutPredictor


from transformers import AutoTokenizer, AutoModelForCausalLM
from surya.recognition import RecognitionPredictor


from tqdm import tqdm
import time
import lmstudio as lms

/home/duckq1u/miniconda3/envs/OCR2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SERVER_API_HOST = "localhost:8080"

# This must be the *first* convenience API interaction (otherwise the SDK
# implicitly creates a client that accesses the default server API host)
lms.configure_default_client(SERVER_API_HOST)


model = lms.llm("qwen2.5-vl-7b-instruct", config={ "seed": 5296672349136066156,"flashAttention": True, "evalBatchSize": 5024, "contextLength": 7024})
# Get current torch seed number 
torch_seed = torch.initial_seed()
print(f"Current torch seed: {torch_seed}")

Current torch seed: 5296672349136066156


## Extractor text from image

In [3]:
format_contact = {
    "1a": [
        "1. Người yêu cầu đăng ký", 
        "2. Hợp đồng bảo đảm",
        "3. Bên bảo đảm",
        "4. Bên nhận bảo đảm",
    ],
    "1b": [
        "1. Người yêu cầu cấp bản sao/Applicant",
        "2. Yêu cầu cấp bản sao Văn bản chứng nhận đăng ký/Request for issuance of copy of registration certificate"
    ],
    "1c": [
        "1. Thông tin chung/General information",
        "2. Hợp đồng bảo đảm/Security agreement"
    ]
}

In [ ]:
def draw_bounding_boxes(image, predictions, is_layout=False):
    draw = ImageDraw.Draw(image)
    # Rounded rectangle parameters
    border_radius = 0  # Corner radius
    outline_color = "red"  # Box color
    outline_width = 1  # Box thickness
    try:
        if is_layout:
            predictions = predictions[0].bboxes
        else:
            predictions = predictions[0].text_lines

        paragraph = ""

        # Loop through each set of coordinates
        for coords in predictions:
            # Extract x and y values
            x_values = [x for x, y in coords.polygon]
            y_values = [y for x, y in coords.polygon]

            # Calculate bounding box
            min_x = min(x_values)
            max_x = max(x_values)
            min_y = min(y_values)
            max_y = max(y_values)

            # Draw the rounded rectangle
            draw.rounded_rectangle(
                [(min_x, min_y), (max_x, max_y)],
                radius=border_radius,
                outline=outline_color,
                width=outline_width,
            )
            if is_layout is False:
                paragraph += coords.text + "\n"
    except Exception as e:
        print('Out of index')
        return False, False
    return image, paragraph


def rounding_box(image, is_layout=False, is_list=False):
    image_cp = image.copy()
    if is_layout:
        layout_predictor = LayoutPredictor()
        detection_predictor = DetectionPredictor()
        predictions = layout_predictor([image_cp])
    else:
        langs = [
            "en"
        ]  # Replace with your languages or pass None (recommended to use None)
        recognition_predictor = RecognitionPredictor()
        detection_predictor = DetectionPredictor()
        if is_list:
            predictions = recognition_predictor(
                image_cp, langs=langs, det_predictor=detection_predictor
            )
        else:
            predictions = recognition_predictor(
                [image_cp], det_predictor=detection_predictor
            )

    image_output, paragraph = draw_bounding_boxes(image_cp, predictions, is_layout)
    return image_output, paragraph

def norm_text(text):
    # remove :, punctuation, and special characters
    text = re.sub(r"[^\w\s]", " ", text)  # replace punctuation and special characters with space
    text = re.sub(r"\s+", " ", text)  # replace multiple spaces with a single space
    return text.strip()

def extract_text_from_pdf(pdf_path, form_code, is_layout=False):
    images = convert_from_path(pdf_path)
    paragraphs = []
    images_list = []
    for image in tqdm(images, desc="Processing pages"):
        image_output, paragraph_output = rounding_box(image, is_layout)
        
        if image_output is False: continue
        count = 0
        
        
        for form in format_contact[form_code]:
            if norm_text(form).lower() in norm_text(paragraph_output).lower():
                count += 1
                
        # BUG: remove an images if it does not contain enough keywords
        if count < len(format_contact[form_code]): continue

        # process the paragraph for lmstudio
        paragraph = f"\n<page {len(paragraphs) + 1}>\n"
        paragraph += paragraph_output + f"\n</page {len(paragraphs) + 1}>\n"
        paragraphs.append(paragraph)
        
        # process the image for lmstudio
        image_output.save('./temp.jpg', 'JPEG')
        image_handle = lms.prepare_image(
            "./temp.jpg",
        )
        images_list.append(image_handle)

    return paragraphs, images_list

In [5]:
def create_prompt(pdf_path, form_code):
    raw_text, images_list = extract_text_from_pdf(pdf_path=pdf_path, form_code=form_code, is_layout=False)

    feature_text = ""
    for text in raw_text:
        feature_text += text
        
    keywords = ""
    json_form = ""
    
    for index, code in enumerate(format_contact[form_code], start=1):
        if code is dict:
            # So i do like below to handle the case where the code is a dict
            # because some use case user want to extract an information its 
            # in a huge paragraph as well as a noise information surrounding it
            # so i will extract the information and put it in a single value
            keywords += f"\n{index}. {code}\n"
            json_form += f'\n"{code}": <information and just have one value>,\n'
        else:
            keywords += f"\n{index}. {code}\n"
            json_form += f'\n"{code}": [ <extracted and related information that can have one or multiple. Pay attention to the types of information checked box and extract the checked information. All <information> inside the JSON must have a KEY and must not stand alone> ],\n'
    
    prompt = f"""
You are an expert in administrative document processing

<Raw text from image> 
{feature_text}
</Raw text from image>


<Notes>
1. Only output the required extracted information.
2. Related information is often located near the extracted information and must also be captured.
3. Perform detailed analysis and do not omit any information.
4. Pay attention to the types of information checked box and extract the checked information.
</Notes>

<Requirements>
Keyword list:
{keywords}

Step 1: Extract information from the IMAGE, using KEYWORDS as markers, starting from left to right and top to bottom until encountering another KEYWORD or unrelated information.

Step 1.1: If the image contains table information related to the any KEYWORD, analyze the table as a list for subsequent retrieval, then load it into the JSON.

Step 2: Extract information from the RAW TEXT FROM IMAGE using KEYWORDS as markers, starting from left to right and top to bottom until encountering another KEYWORD or unrelated information.

Step 3: Use information extracted from IMAGE to correct spelling errors in the RAW TEXT FROM IMAGE.

Step 4: Re-analyze the information and correct the spelling of the information before extracting.

Step 5: Load the data into the following JSON form:
```JSON
{{
{json_form}
}}
```
While filling in the JSON, ensure that:
1. KEYs with the same or equivalent meanings can be grouped into a list.
2. Do not alter the format or structure of the JSON.
3. If the <information> is in the form of a table, it should be stored in a list with column headers as titles.
4. Merge objects with the same value.
5. Remove duplicate data fields before output.

Step 5: Output in JSON file format.
</Requirements>"""

    return  prompt, images_list

In [ ]:
folder_path = "/home/duckq1u/Documents/obsidian_aio/Notebook/Dự án/OCR anh hiếu/data_test"
total_time = 0

for folder in os.listdir(folder_path):
    start_time = time.time()
    folder_path_ = os.path.join(folder_path, folder)
    if not os.path.isdir(folder_path_): continue
    
    print("///////"*40)
    print(f"== form {folder} ==")
    for file in os.listdir(folder_path_):
        if not file.endswith(".pdf"): continue
        
        pdf_path = os.path.join(folder_path_, file)

        print(f"Đang xử lý file: {file}")
        
        prompt, images_list = create_prompt(pdf_path=pdf_path, form_code=file.split("_")[0])
        
        if len(images_list) == 0:
            print("No images found in the PDF.")
            continue
        
        # Gửi request chat
        chat = lms.Chat("You are an expert in administrative document processing.")
        chat.add_user_message(prompt, images=images_list)
        
        prediction = model.respond(chat)
        
        duration = time.time() - start_time
        total_time += duration
        
        print("***" * 40)
        print("== Kết quả dự đoán ==")
        print(prediction)        
        print(f"Thời gian xử lý: {duration:.2f}s")
        print("***" * 40)
    print("///////"*40)

////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////////
== form 1a ==
Đang xử lý file: 1a_2.pdf


Processing pages: 100%|██████████| 6/6 [01:01<00:00, 10.31s/it]


************************************************************************************************************************
== Kết quả dự đoán ==
```json
{
    "1. Người yêu cầu đăng ký": [
        {
            "Bên nhận bảo đảm": false,
            "Quản tài viên/Doanh nghiệp quản lý, thanh lý tài sản": false,
            "Chi nhánh của pháp nhân, người đại diện": true,
            "Họ và tên đầy đủ đối với cá nhân/tên đầy đủ đối với tổ chức (viết chữ IN HOA)": "LÊ THỊ MAI",
            "Địa chỉ để cơ quan đăng ký liên hệ khi cần thiết": "456 Phố Huế, Hai Bà Trưng, Hà Nội"
        }
    ],
    "2. Hợp đồng bảo đảm": [
        {
            "HĐ567/2025 số (nếu có)": "",
            "Thời điểm có hiệu lực": "ngày 13 tháng 06 năm 2025"
        }
    ],
    "3. Bên bảo đảm": [
        {
            "Bên nhận bảo đảm": false,
            "Họ và tên đầy đủ đối với cá nhân/tên đầy đủ đối với tổ chức (viết chữ IN HOA)": "LÊ THỊ MAI",
            "Địa chỉ": "456 Phố Huế, Hai Bà Trưng, Hà Nội",
 

Processing pages: 100%|██████████| 6/6 [00:18<00:00,  3.15s/it]


No images found in the PDF.
Đang xử lý file: 1a_1.pdf


...

Out of index


{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': ' \"\",\\n', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '           ', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': ' \"', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'messa

KeyboardInterrupt: 

{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '1', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '2', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '3', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}


{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '4', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '5', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragment', 'fragment': {'content': '6', 'tokensCount': 1, 'containsDrafted': False, 'reasoningType': 'none'}}} for already closed channel", "ws_url": "ws://localhost:8080/llm"}
{"channel_id": 5, "event": "Received unhandled message {'type': 'channelSend', 'channelId': 5, 'message': {'type': 'fragm